In [1]:
import pandas as pd
from datetime import timedelta


In [2]:
df = pd.read_csv("outputs/player_daily_innings_impact.csv")

df["date"] = pd.to_datetime(df["date"])


In [3]:
df = df[
    [
        "player",
        "match_id",
        "date",
        "league",
        "innings",
        "impact_type",
        "innings_value"
    ]
].copy()


In [4]:
df = df.sort_values(
    ["player", "date", "match_id", "innings"]
).reset_index(drop=True)


In [5]:
def compute_last_n_innings_form(
    df,
    n_innings,
    league_scope="ALL"
):
    rows = []

    if league_scope == "ALL":
        base_df = df.copy()
        league_label = "ALL"
    else:
        base_df = df[df["league"] == league_scope].copy()
        league_label = league_scope

    grouped = base_df.groupby(
        ["player", "innings", "impact_type"]
    )

    for (player, innings, impact_type), g in grouped:
        g = g.sort_values("date")

        for i in range(len(g)):
            window = g.iloc[max(0, i - n_innings + 1): i + 1]

            rows.append({
                "player": player,
                "as_of_date": g.iloc[i]["date"],
                "innings_filter": innings,
                "impact_type": impact_type,
                "league_scope": "ALL" if league_scope == "ALL" else "LEAGUE",
                "league": league_label,
                "window_type": "LAST_N_INNINGS",
                "window_value": n_innings,
                "form_value": window["innings_value"].sum(),
                "innings_count": len(window),
                "matches_count": window["match_id"].nunique()
            })

    return pd.DataFrame(rows)


In [6]:
def compute_last_n_days_form(
    df,
    n_days,
    league_scope="ALL"
):
    rows = []

    if league_scope == "ALL":
        base_df = df.copy()
        league_label = "ALL"
    else:
        base_df = df[df["league"] == league_scope].copy()
        league_label = league_scope

    grouped = base_df.groupby(
        ["player", "innings", "impact_type"]
    )

    for (player, innings, impact_type), g in grouped:
        g = g.sort_values("date")

        for _, row in g.iterrows():
            cutoff = row["date"] - timedelta(days=n_days)

            window = g[g["date"] >= cutoff]

            rows.append({
                "player": player,
                "as_of_date": row["date"],
                "innings_filter": innings,
                "impact_type": impact_type,
                "league_scope": "ALL" if league_scope == "ALL" else "LEAGUE",
                "league": league_label,
                "window_type": "LAST_N_DAYS",
                "window_value": n_days,
                "form_value": window["innings_value"].sum(),
                "innings_count": len(window),
                "matches_count": window["match_id"].nunique()
            })

    return pd.DataFrame(rows)


In [7]:
INNINGS_WINDOWS = [5, 10]
DAY_WINDOWS = [180 , 360]


In [8]:
frames = []

for n in INNINGS_WINDOWS:
    frames.append(
        compute_last_n_innings_form(df, n, league_scope="ALL")
    )

for d in DAY_WINDOWS:
    frames.append(
        compute_last_n_days_form(df, d, league_scope="ALL")
    )


In [9]:
for league in df["league"].dropna().unique():
    for n in INNINGS_WINDOWS:
        frames.append(
            compute_last_n_innings_form(df, n, league_scope=league)
        )

    for d in DAY_WINDOWS:
        frames.append(
            compute_last_n_days_form(df, d, league_scope=league)
        )


In [10]:
player_form_rolling = pd.concat(
    frames,
    ignore_index=True
)


In [11]:
pd.crosstab(
    player_form_rolling["innings_filter"],
    player_form_rolling["impact_type"]
)


impact_type,RPA,WPA
innings_filter,,
1,650128,0
2,0,629496


In [12]:
(player_form_rolling["innings_count"] == 0).sum()


np.int64(0)

In [13]:
player_form_rolling.head()


,player,as_of_date,innings_filter,impact_type,league_scope,league,window_type,window_value,form_value,innings_count,matches_count
0,A Aravinddaraj,2019-02-25,1,RPA,ALL,ALL,LAST_N_INNINGS,5,-15.652720,1,1
1,A Aravinddaraj,2019-11-12,1,RPA,ALL,ALL,LAST_N_INNINGS,5,-13.208107,2,2
2,A Aravinddaraj,2019-11-15,1,RPA,ALL,ALL,LAST_N_INNINGS,5,-25.786156,3,3
3,A Aravinddaraj,2021-01-17,1,RPA,ALL,ALL,LAST_N_INNINGS,5,-18.923628,4,4
4,A Aravinddaraj,2021-01-19,1,RPA,ALL,ALL,LAST_N_INNINGS,5,-40.772460,5,5


In [14]:
OUTPUT_PATH = "outputs/player_form_rolling.csv"

player_form_rolling.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Saved rolling form table to: {OUTPUT_PATH}")


Saved rolling form table to: outputs/player_form_rolling.csv


In [15]:
def get_player_form(
    df,
    player,
    innings,
    league="ALL",
    window_type="LAST_N_INNINGS",
    window_value=5
):
    """
    Returns the latest form snapshot for a player
    under a specific context.
    """

    sub = df[
        (df["player"] == player) &
        (df["innings_filter"] == innings) &
        (df["window_type"] == window_type) &
        (df["window_value"] == window_value)
    ]

    if league == "ALL":
        sub = sub[sub["league_scope"] == "ALL"]
    else:
        sub = sub[
            (sub["league_scope"] == "LEAGUE") &
            (sub["league"] == league)
        ]

    if sub.empty:
        return None

    latest = sub.sort_values("as_of_date").iloc[-1]

    return {
        "player": player,
        "innings": innings,
        "league": league,
        "form_value": round(latest["form_value"], 3),
        "innings_count": int(latest["innings_count"]),
        "matches_count": int(latest["matches_count"]),
        "as_of_date": latest["as_of_date"]
    }


In [19]:
get_player_form(
    player_form_rolling,
    player="V Kohli",
    innings=1,
    league="IPL",
    window_value=5
)


{'player': 'V Kohli',
 'innings': 1,
 'league': 'IPL',
 'form_value': np.float64(61.162),
 'innings_count': 5,
 'matches_count': 5,
 'as_of_date': Timestamp('2025-06-03 00:00:00')}

In [20]:
def compare_players_form(
    df,
    player_a,
    player_b,
    innings,
    league="ALL",
    window_value=5
):
    fa = get_player_form(df, player_a, innings, league, window_value=window_value)
    fb = get_player_form(df, player_b, innings, league, window_value=window_value)

    return {
        "player_a": fa,
        "player_b": fb
    }


In [22]:
compare_players_form(player_form_rolling,
                     'V Kohli',
                      'Shubman Gill',
                       2,
                        "IPL",5  )

{'player_a': {'player': 'V Kohli',
  'innings': 2,
  'league': 'IPL',
  'form_value': np.float64(0.708),
  'innings_count': 5,
  'matches_count': 5,
  'as_of_date': Timestamp('2025-05-29 00:00:00')},
 'player_b': {'player': 'Shubman Gill',
  'innings': 2,
  'league': 'IPL',
  'form_value': np.float64(0.404),
  'innings_count': 5,
  'matches_count': 5,
  'as_of_date': Timestamp('2025-05-30 00:00:00')}}

In [23]:
def top_players_by_form(
    df,
    innings,
    league="ALL",
    window_value=5,
    top_n=10,
    min_innings=3
):
    sub = df[
        (df["innings_filter"] == innings) &
        (df["window_type"] == "LAST_N_INNINGS") &
        (df["window_value"] == window_value) &
        (df["innings_count"] >= min_innings)
    ]

    if league == "ALL":
        sub = sub[sub["league_scope"] == "ALL"]
    else:
        sub = sub[
            (sub["league_scope"] == "LEAGUE") &
            (sub["league"] == league)
        ]

    latest = (
        sub.sort_values("as_of_date")
        .groupby("player", as_index=False)
        .tail(1)
    )

    return (
        latest
        .sort_values("form_value", ascending=False)
        .head(top_n)
        [["player", "form_value", "innings_count", "matches_count"]]
    )


In [25]:
top_players_by_form(
    player_form_rolling,
    innings=1,
    league="IPL",
    window_value=5
)


,player,form_value,innings_count,matches_count
714613,MR Marsh,101.778135,5,5
721238,Shubman Gill,85.879050,5,5
709588,H Klaasen,79.738183,5,5
710538,J Fraser-McGurk,64.818578,5,5
708678,DP Conway,64.456634,5,5
710475,Ishan Kishan,62.494970,5,5
709853,HM Amla,61.905646,5,5
722136,V Kohli,61.162472,5,5
707086,B Sai Sudharsan,58.746830,5,5
707903,D Brevis,58.716297,5,5


In [26]:
def player_form_trend(
    df,
    player,
    innings,
    league="ALL",
    window_value=5
):
    sub = df[
        (df["player"] == player) &
        (df["innings_filter"] == innings) &
        (df["window_type"] == "LAST_N_INNINGS") &
        (df["window_value"] == window_value)
    ]

    if league == "ALL":
        sub = sub[sub["league_scope"] == "ALL"]
    else:
        sub = sub[
            (sub["league_scope"] == "LEAGUE") &
            (sub["league"] == league)
        ]

    return sub.sort_values("as_of_date")[
        ["as_of_date", "form_value", "innings_count"]
    ]


In [30]:
player_form_trend(
    player_form_rolling,
    player="Rashid Khan",
    innings=1,
    league="IPL",
    window_value=5
)


,as_of_date,form_value,innings_count
718553,2017-04-09,18.281218,1
718554,2017-04-12,15.455682,2
718555,2017-04-15,26.744505,3
718556,2017-05-06,39.542126,4
718557,2017-05-08,48.301961,5
...,...,...,...
718628,2025-05-06,-59.486351,5
718629,2025-05-18,-31.542199,5
718630,2025-05-22,-41.287648,5
718631,2025-05-25,-61.265118,5


In [ ]:
def shortlist_players_current_form(
    df,
    innings,
    league="ALL",
    window_value=5,
    min_innings=3,
    top_n=25,
    active_within_days=365
):
    """
    Returns top in-form ACTIVE players only.
    """

    sub = df[
        (df["innings_filter"] == innings) &
        (df["window_type"] == "LAST_N_INNINGS") &
        (df["window_value"] == window_value) &
        (df["innings_count"] >= min_innings)
    ].copy()

    # League filter
    if league == "ALL":
        sub = sub[sub["league_scope"] == "ALL"]
    else:
        sub = sub[
            (sub["league_scope"] == "LEAGUE") &
            (sub["league"] == league)
        ]

    #  KEEP ONLY LATEST SNAPSHOT PER PLAYER
    latest_per_player = (
        sub.sort_values("as_of_date")
        .groupby("player", as_index=False)
        .tail(1)
    )

    #  REMOVE RETIRED / INACTIVE PLAYERS
    latest_date = latest_per_player["as_of_date"].max()
    cutoff_date = latest_date - pd.Timedelta(days=active_within_days)

    latest_per_player = latest_per_player[
        latest_per_player["as_of_date"] >= cutoff_date
    ]

    return (
        latest_per_player
        .sort_values("form_value", ascending=False)
        .head(top_n)
        [["player", "form_value", "innings_count", "matches_count", "as_of_date"]]
    )


In [35]:
shortlist_players_current_form(
    player_form_rolling,
    innings=1,
    league="IPL",
    window_value=5,
    active_within_days=365
)


,player,form_value,innings_count,matches_count,as_of_date
714613,MR Marsh,101.778135,5,5,2025-05-27
721238,Shubman Gill,85.879050,5,5,2025-05-02
709588,H Klaasen,79.738183,5,5,2025-05-25
710538,J Fraser-McGurk,64.818578,5,5,2025-04-05
708678,DP Conway,64.456634,5,5,2025-05-25
710475,Ishan Kishan,62.494970,5,5,2025-05-25
722136,V Kohli,61.162472,5,5,2025-06-03
707086,B Sai Sudharsan,58.746830,5,5,2025-05-02
707903,D Brevis,58.716297,5,5,2025-05-25
706452,Abishek Porel,52.725324,5,5,2025-05-18
